# 5 · Run All Scenarios (sweep) on FABRIC

Execute **every** scenario in `validation/scenarios/` end-to-end on the slice — single
scenarios plus the expanded `sweep_distance` (6 pts) and `sweep_attenuation` (9 pts) — and
for each point run the full 4-way cross-validation (QFabric measured + QFabric-sim +
SeQUeNCe + NetSquid). Then plot QBER and secure key rate vs distance/attenuation.

> ⏳ **This is the long one.** ~18 points × (switch reconfigure + BB84 + 3 simulator runs)
> ≈ 30–60+ min of slice time. Results are written to `results/all_scenarios.json` after
> **every** point, so an interruption keeps what's done.

**Prerequisite:** `01_setup_slice` (slice up, data-plane IPs) completed. The simulator
envs are built in Step 1 below (one-time).

### At a glance
- **Purpose:** batch-run **every** scenario in `validation/scenarios/` (singles + expanded sweeps) and produce the QBER/key-rate-vs-distance/attenuation figures.
- **Inputs:** `SLICE_NAME`, NetSquid creds (Step 1).
- **Outputs:** `results/all_scenarios.json` (rewritten after every point) + sweep figures + an all-points table.
- **Runs on / runtime:** FABRIC JupyterHub; **~30–60+ min** (18 points × switch reconfigure + BB84 + 3 simulator runs). Set `cross_validate=False` for a faster BB84-only sweep.
- **If something fails:** the sweep is resumable (partial `all_scenarios.json` is kept) and one bad point won't abort the rest; high-loss points may legitimately show the measured backend as SKIPPED while the simulators still populate the row.

## Configuration (match notebook 1)

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## Step 1 — Build simulator envs on the nodes (one-time)
Set your netsquid.org credentials first so NetSquid installs on the bob node.

In [ ]:
# os.environ['NETSQUID_USER'] = '...'
# os.environ['NETSQUID_PASS'] = '...'
df.setup_sim_envs(slice_obj)

## Step 2 — Run every scenario (this takes a while)
Reconfigures the switch loss threshold per point, runs BB84, then the 4-way comparison.
Set `cross_validate=False` for a faster BB84-only sweep.

In [ ]:
rows = df.run_all_scenarios_on_fabric(slice_obj, cross_validate=True)
print(f'\nCompleted {len(rows)} scenario points.')

## Step 3 — Sweep figures: QBER & key rate vs distance / attenuation
One line per backend; loads `results/all_scenarios.json` (so this cell also works on a
previously-saved sweep without re-running Step 2).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
rows = json.loads((PROJECT_DIR / 'results' / 'all_scenarios.json').read_text())

def _ok(b):
    return (not b.get('extra', {}).get('error')) and b.get('sifted_bits', 0) > 0

def plot_group(group, xkey, xlabel):
    pts = sorted([r for r in rows if r['group'] == group], key=lambda r: r[xkey])
    if not pts:
        print(f'(no points for {group})'); return
    platforms = []
    for r in pts:
        for b in r['backends']:
            if b['platform'] not in platforms: platforms.append(b['platform'])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for p in platforms:
        xs, q, s = [], [], []
        for r in pts:
            b = next((b for b in r['backends'] if b['platform'] == p and _ok(b)), None)
            if b:
                xs.append(r[xkey]); q.append(b['qber'] * 100); s.append(b.get('secure_key_rate', 0))
        if xs:
            axes[0].plot(xs, q, marker='o', label=p)
            axes[1].plot(xs, s, marker='o', label=p)
    axes[0].set(xlabel=xlabel, ylabel='QBER (%)', title=f'QBER vs {xlabel}')
    axes[1].set(xlabel=xlabel, ylabel='Secure key rate (bits/photon)', title=f'Key rate vs {xlabel}')
    for ax in axes: ax.grid(alpha=0.3); ax.legend()
    fig.suptitle(group); plt.tight_layout(); plt.show()

plot_group('sweep_distance', 'distance_km', 'distance (km)')
plot_group('sweep_attenuation', 'attenuation_db_per_km', 'attenuation (dB/km)')

## Step 4 — All-scenarios summary table

Every point (singles + sweep points) with each backend's QBER. A blank/`NaN`
means that backend was SKIPPED for that point (e.g. a measured run that produced
no key at high loss, or an uninstalled simulator).

In [ ]:
import pandas as pd
tbl = []
for r in rows:
    row = {'group': r['group'], 'scenario': r['scenario'],
           'distance_km': r['distance_km'], 'atten': r['attenuation_db_per_km']}
    for b in r['backends']:
        row[b['platform']] = round(b['qber'], 4) if _ok(b) else None
    tbl.append(row)
df = pd.DataFrame(tbl)
print(f'{len(df)} scenario points')
df

---
All scenarios executed. `results/all_scenarios.json` holds every point × backend for
further analysis or for the paper figures.